# Retrieval: Query Transformation (HyDE, Multi-Query, Step-Back)

Users rarely phrase a question the way the answer is written in the corpus. **Query transformation** rewrites the question *before* retrieval so it matches the index better:

| Strategy | Idea | Helps with |
|---|---|---|
| **HyDE** | Embed an LLM-written *hypothetical answer* instead of the question | Vocabulary mismatch |
| **Multi-Query** | Retrieve for several rephrasings, then merge | Recall |
| **Step-Back** | Also retrieve for a more *general* version of the question | Over-specific questions |

We compare them against a plain dense baseline on HotpotQA (indexed in `6.5-Storage-Vector-Databases.ipynb`).

In [ ]:
%pip install -q langchain langchain-qdrant langchain-huggingface langchain-openai qdrant-client sentence-transformers rapidfuzz tqdm

## 1. Setup

Everything comes from `config.yaml` through the shared helper `rag_common.py`: the Qdrant collection, the HotpotQA questions, and the LLM.

In [ ]:
from statistics import mean

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from tqdm.auto import tqdm

import rag_common as rc

cfg = rc.load_config()
rc.check_services(cfg)
qa_pairs, _ = rc.load_hotpotqa(cfg)
client, embeddings, doc_store = rc.connect_qdrant(cfg)
llm_name, llm = next(iter(rc.load_llms(cfg).items()))

base_retriever = doc_store.as_retriever(search_kwargs={"k": 3})
print(f"Collection: {cfg['hotpotqa']['collection']} ({client.get_collection(cfg['hotpotqa']['collection']).points_count} vectors) | LLM: {llm_name}")

eval_set = rc.eval_set(qa_pairs)   # the same fixed sample every RAG notebook scores on
print(f"Evaluation set: {len(eval_set)} questions")

## 2. Evaluation Harness

Each strategy is a function `question -> (docs, answer)`. One shared loop scores them all with the same metrics: **hit** (answer text appears in the retrieved context) and **token F1** (answer quality).

In [ ]:
def run_strategy(label, strategy):
    """Run a strategy over eval_set and score retrieval (hit) and answer quality (F1)."""
    rows = []
    for ex in tqdm(eval_set, desc=label):
        docs, prediction, *note = strategy(ex["question"])
        rows.append({
            "question": ex["question"], "reference": ex["answer"], "prediction": prediction,
            "note": note[0] if note else "",
            "hit": rc.retrieval_hit(docs, ex["answer"]),
            "f1": rc.token_f1(prediction, ex["answer"]),
        })
    return rows


def report(label, rows):
    print(f"{label:<14} Hit rate: {mean(r['hit'] for r in rows)*100:5.1f}%   Token F1: {mean(r['f1'] for r in rows)*100:5.1f}%")


def baseline(question):
    docs = base_retriever.invoke(question)
    return docs, rc.answer_with_docs(question, docs, llm)


results = {"Baseline": run_strategy("Baseline", baseline)}
report("Baseline", results["Baseline"])

## 3. HyDE — Hypothetical Document Embeddings

Ask the LLM to *write* a plausible answer passage, then embed **that** instead of the question. A generated passage shares vocabulary and style with real documents far better than a short question does. The hypothesis may be factually wrong; it only serves as a retrieval query.

```
Question → LLM → hypothetical passage → embed → retrieve → LLM → answer
```

In [ ]:
hyde_prompt = ChatPromptTemplate.from_template("""
Write a short factual passage (2-3 sentences) that would directly answer the following question.
Do NOT say you don't know — write your best guess as if it were a Wikipedia excerpt.

Question: {question}

Passage:
""")
hyde_chain = hyde_prompt | llm | StrOutputParser()


def hyde(question, k=3):
    hypothesis = hyde_chain.invoke({"question": question})
    docs = doc_store.similarity_search(hypothesis, k=k)   # embed the hypothesis, not the question
    return docs, rc.answer_with_docs(question, docs, llm), hypothesis


sample_q = eval_set[0]["question"]
print("Question:", sample_q)
print("\nHypothetical passage:\n", hyde_chain.invoke({"question": sample_q}))

In [ ]:
results["HyDE"] = run_strategy("HyDE", hyde)
report("HyDE", results["HyDE"])

## 4. Multi-Query Retrieval

A single embedding captures one phrasing. Generate *n* alternatives, retrieve for each, and take the de-duplicated union. This improves **recall** at the cost of extra LLM and retrieval calls.

```
Question → LLM → [Q1, Q2, Q3] → retrieve each → union → LLM → answer
```

In [ ]:
multiquery_prompt = ChatPromptTemplate.from_template("""
Generate {n} different ways to ask the following question, each from a slightly different angle or vocabulary.
Output ONLY the rephrased questions, one per line, with no numbering or extra text.

Original question: {question}

Rephrasings:
""")
multiquery_chain = multiquery_prompt | llm | StrOutputParser()


def multi_query(question, n=3, k=3):
    raw = multiquery_chain.invoke({"question": question, "n": n})
    variants = [q.strip() for q in raw.splitlines() if q.strip()][:n]

    seen, docs = set(), []
    for query in [question, *variants]:                   # always keep the original
        for doc in base_retriever.invoke(query):
            key = doc.page_content[:100]                  # de-duplicate on content prefix
            if key not in seen:
                seen.add(key)
                docs.append(doc)
    docs = docs[: 2 * k]                                  # cap context size
    return docs, rc.answer_with_docs(question, docs, llm), " | ".join(variants)


print("Variants for:", sample_q)
print(multiquery_chain.invoke({"question": sample_q, "n": 3}))

In [ ]:
results["Multi-Query"] = run_strategy("Multi-Query", multi_query)
report("Multi-Query", results["Multi-Query"])

## 5. Step-Back Prompting

Specific questions often miss because the corpus states the *general* fact. Step-Back (Zheng et al., 2023) first rewrites the question into a broader one, retrieves background for it, then answers the original question using both contexts.

```
"Who scored the winning goal in the 1966 final?"  →  step-back: "What happened in the 1966 World Cup?"
```

In [ ]:
stepback_prompt = ChatPromptTemplate.from_template("""
Rewrite the question below into a broader, more general question that captures the underlying
topic and could be answered by a general encyclopedic passage.

Example:
  Original:  "What temperature does water boil at on Mount Everest?"
  Step-back: "How does altitude affect the boiling point of water?"

Original:  {question}
Step-back:
""")
stepback_answer_prompt = ChatPromptTemplate.from_template("""
Use the background context and the specific context to answer the question.
If the answer is not available, say so.

Background context:
{background}

Specific context:
{specific}

Question: {question}

Reply with only the answer itself (a few words, no explanation, no full sentence).

Answer:
""")


def step_back(question):
    broad_question = (stepback_prompt | llm | StrOutputParser()).invoke({"question": question}).strip()
    background_docs = base_retriever.invoke(broad_question)
    specific_docs = base_retriever.invoke(question)
    answer = (stepback_answer_prompt | llm | StrOutputParser()).invoke({
        "background": rc.format_docs(background_docs),
        "specific": rc.format_docs(specific_docs),
        "question": question,
    })
    return specific_docs, answer, broad_question

In [ ]:
results["Step-Back"] = run_strategy("Step-Back", step_back)
report("Step-Back", results["Step-Back"])

## 6. Comparison

Small sample, so treat differences of a few points as noise. The point is to see *where* each strategy changes retrieval (hit) versus only the final answer (F1).

In [ ]:
print(f"Query transformation on HotpotQA (n={len(eval_set)}, LLM: {llm_name})\n")
print(f"{'Strategy':<14}{'Hit %':>8}{'F1 %':>8}")
print("-" * 30)
for label, rows in results.items():
    print(f"{label:<14}{mean(r['hit'] for r in rows)*100:>8.1f}{mean(r['f1'] for r in rows)*100:>8.1f}")

print("\nStep-back rewrites:")
for r in results["Step-Back"][:4]:
    print(f"  {r['question']}\n    → {r['note']}")

In [ ]:
for label, rows in results.items():
    rc.record_result("Dense top-3 (baseline)" if label == "Baseline" else label,
                     "retrieval" if label == "Baseline" else "query-transform", rows=rows)

## Summary

- **HyDE** fixes vocabulary mismatch but can drift if the LLM hallucinates a misleading hypothesis.
- **Multi-Query** raises recall; deduplicate and cap the merged context.
- **Step-Back** helps over-specific questions; costs a second retrieval and a longer prompt.
- All three add LLM calls *before* retrieval — measure the gain against latency and cost.

Next: `6.9-Retrieval-Rerank-and-Compression.ipynb` improves what happens *after* retrieval.